## Recomender System Using PyTorch

This is a tutorial for building a movie recommender system using PyTorch, taken from [article](https://pureai.substack.com/p/recommender-systems-with-pytorch). Our focus will be on the MovieLens dataset, a compact yet insightful collection of movie ratings, perfect for demonstrating the capabilities of a recommender system.

### Defining the Dataset and Relevant Class

In this section, we dive into the practical steps of setting up our recommender system. We'll be using the widely recognized MovieLens dataset, specifically a file named ratings.csv, which contains user ratings for various movies. This file includes features like userId, movieId, rating (ranging from 1.0 to 5.0), and timestamp.

In [3]:
import pandas as pd
DATA_DIR = '../data/ml-latest-small/'

df = pd.read_csv(DATA_DIR + "ratings.csv")
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


next create a custom dataset class, to handle data for PyTorch. this class will prepare the dataset for training and validation processes.

In [1]:
from torch.utils.data import Dataset
import torch

class MovieLensDataset(Dataset):
  """
  The Movie Lens Dataset class. this class prepares the dataset for training and validation
  """
  def __init__(self, users, movies, ratings):
    """
    Initializes the dataset object with user, movie, and rating data.
    """
    super().__init__()
    self.users = users
    self.movies = movies
    self.ratings = ratings
  
  def __len__(self):
    """
    Returns the total number of samples in the dataset.
    """
    return len(self.users)
  
  def __getitem__(self, index):
    """
    Retrieves a sample from the dataset at the specified index
    """
    users = self.users[index]
    movies = self.movies[index]
    ratings = self.ratings[index]
    
    return {
      "users": torch.tensor(users, dtype=torch.long),
      "movies": torch.tensor(movies, dtype=torch.long),
      "ratings": torch.tensor(ratings, dtype=torch.float),
    }

In [2]:
import torch
from torch import nn

class RecommendationSystemModel(nn.Module):
  def __init__(self, num_users, num_movies, embedding_size=256, hidden_dim=256, dropout_rate=0.2):
    super(RecommendationSystemModel, self).__init__()
    self.num_users=num_users
    self.num_movies=num_movies
    self.embedding_size=embedding_size
    self.hidden_dim=hidden_dim
    
    # Embedding layers
    self.user_embedding = nn.Embedding(
      num_embeddings=self.num_users, embedding_dim=self.embedding_size
    )
    self.movie_embedding = nn.Embedding(
      num_embeddings=self.num_movies, embedding_dim=self.embedding_size
    )
    
    # Hidden layers
    self.fc1 = nn.Linear(2 * self.embedding_size, self.hidden_dim)
    self.fc2 = nn.Linear(self.hidden_dim, 1)
    
    # Dropout Layer
    self.dropout = nn.Dropout(p=dropout_rate)
    
    # Activation function
    self.relu = nn.ReLU()
    
  def forward(self, users, movies):
    # Embeddings 
    user_embedded = self.user_embedding(users)
    movie_embedded = self.movie_embedding(movies)
    
    # Conctentae user and movies mebeddings
    combined = torch.cat([user_embedded, movie_embedded], dim=1)
    
    # Pass through hiddenlyers with ReLU activation and dropout
    x = self.relu(self.fc1(combined))
    x = self.dropout(x)
    output = self.fc2(x)
    
    return output

In [4]:
# preprocessing and encoding
from sklearn import preprocessing

le_user = preprocessing.LabelEncoder()
le_movie = preprocessing.LabelEncoder()
df.userId = le_user.fit_transform(df.userId.values)
df.movieId = le_movie.fit_transform(df.movieId.values)

In [5]:
# splitting dataset
from sklearn import model_selection

df_train, df_val = model_selection.train_test_split(
  df, test_size=0.1, random_state=3, stratify=df.rating.values
)

train_dataset = MovieLensDataset(
  users=df_train.userId.values,
  movies=df_train.movieId.values,
  ratings=df_train.rating.values,
)

valid_dataset = MovieLensDataset(
  users=df_val.userId.values,
  movies=df_val.movieId.values,
  ratings=df_val.rating.values,
)

In [6]:
# initialize data loaders
from torch.utils.data import DataLoader

BATCH_SIZE = 32
 
train_loader = DataLoader(
  train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8
)
val_loader = DataLoader(
  valid_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8
)

In [7]:
# define the training loop
device =  torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

recommendation_model = RecommendationSystemModel(
  num_users=len(le_user.classes_),
  num_movies=len(le_movie.classes_),
  embedding_size=128,
  hidden_dim=256,
  dropout_rate=0.1
).to(device)

optimizer = torch.optim.Adam(recommendation_model.parameters(), lr=1e-3)
loss_func = nn.MSELoss()

EPOCHS = 2

cuda


In [8]:
import sys
# function to log progress
def log_progress(epoch, step, total_loss, log_progress_step, data_size, losses):
  avg_loss = total_loss / log_progress_step
  sys.stderr.write(
    f"\r{epoch+1:02d}/{EPOCHS:02d} | Step: {step}/{data_size} | Avg Loss: {avg_loss:<6.9f}"
  )
  sys.stderr.flush()
  losses.append(avg_loss)

In [ ]:
total_loss = 0
log_progress_step = 100
losses = []
train_dataset_size = len(train_dataset)
print(f"Training on {train_dataset_size} samples...")

recommendation_model.train()
for e in range(EPOCHS):
  step_count = 0 # Reset step count at the beginning of each epoch
  for i, train_data in enumerate(train_loader):
    output = recommendation_model(
      train_data["users"].to(device), train_data["movies"].to(device),
    )
    # Reshape the model to match the target's shape
    output = output.squeeze() # remove the singleton dimension
    ratings = (
      train_data["ratings"].to(torch.float32).to(device)
    ) # assuming ratings is already 1D
    
    loss = loss_func(output, ratings)
    total_loss += loss.sum().item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # increment step count by the actul size of the batch
    step_count += len(train_data["users"])
    
    # Check if it's time to log progress
    if (
      step_count % log_progress_step == 0 or i == len(train_loader) - 1
    ):
      log_progress(
        e, step_count, total_loss, log_progress_step, train_dataset_size, losses
      )
      total_loss=0

Training on 90752 samples...


In [ ]:
# RMS (Root Mean Square)f
from sklearn.metrics import mean_squared_error

y_pred = []
y_true = []

recommendation_model.eval()

with torch.no_grad():
  for i, valid_data in enumerate(val_loader):
    output = recommendation_model(
      valid_data["users"].to(device), valid_data["movies"].to(device),
    )
    ratings = (
      valid_data["ratings"].to(torch.float32).to(device)
    )
    y_pred.extend(output.cpu().numpy())
    y_true.extend(ratings.cpu().numpy())


# calculate RMSE
rms = mean_squared_error(y_true, y_pred, squared=False)
print(f"RMSE: {rms:.4f}")